In [111]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
import time
from selenium.common.exceptions import ElementClickInterceptedException
import pandas as pd
import numpy as np


In [8]:
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.page_load_strategy = "eager"   # KEY FIX for Yahoo -- Don’t wait for everything on the page. Continue as soon as the main HTML is ready.


driver = webdriver.Chrome(options=options)
driver.maximize_window()
driver.set_page_load_timeout(60)  #If page loading takes more than 60 seconds, stop waiting and throw an error


#explict wait
wait = WebDriverWait(driver, 30)


# driver = webdriver.Chrome()
driver.maximize_window()


# funtion to check if the web page is fully loaded
def wait_for_the_page_to_load(driver, wait):
    page_title = driver.title
    try:
        wait.until(
            lambda d: d.execute_script("return document.readyState") in ("interactive", "complete")
        )
        time.sleep(3)
    except TimeoutException:
        print(f'The page \"{page_title}\" did not fully load in time.')
    else:
        print(f'The page \"{page_title}\" is ready.')



url = 'https://finance.yahoo.com/'
driver.get(url)
wait_for_the_page_to_load(driver, wait)

# hovering on the market menu

actions = ActionChains(driver)
markets_menu = wait.until(
    EC.presence_of_element_located((By.XPATH, '/html[1]/body[1]/div[2]/header[1]/div[1]/nav[1]/ol[1]/li[3]/a[1]/div[1]'))
)
actions.move_to_element(markets_menu).perform() 

# hovering on the stocks

stocks = wait.until(
    EC.visibility_of_element_located((By.XPATH, '/html[1]/body[1]/div[2]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/a[1]/span[1]'))
)

actions.move_to_element(stocks).perform()


#click on trending tickers

trending = wait.until(
    EC.visibility_of_element_located((By.XPATH, '/html[1]/body[1]/div[2]/header[1]/div[1]/nav[1]/ol[1]/li[3]/ol[1]/li[1]/ol[1]/li[4]/a[1]/span[1]'))
)

actions.move_to_element(trending).click().perform()
time.sleep(2)
wait_for_the_page_to_load(driver, wait)

# Click on the most Active 
most_active = wait.until(
    EC.element_to_be_clickable((By.XPATH, '/html[1]/body[1]/div[2]/div[3]/main[1]/section[1]/section[1]/section[1]/section[1]/section[1]/div[1]/div[1]/div[1]/a[1]'))
)
most_active.click()
time.sleep(2)
wait_for_the_page_to_load(driver, wait)

# scraping the data


page = 1
print(f'Scraping page {page}')
Data = []
while True:
    #scraping
    wait.until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )
    rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
    print(f'Number of rows Page \"{page}\" has is:',len(rows))
    for row in rows:
        values = row.find_elements(By.TAG_NAME, "td")
        Stocks_ = {
            "name": values[1].text,
            "symbol": values[0].text,
            "price": values[3].text,
            "change": values[4].text,
            "volume": values[6].text,
            "market_cap": values[8].text,
            "pr_ratio": values[9].text
        }
        Data.append(Stocks_)


    #click next
    try:
        Next_button = wait.until(
            EC.element_to_be_clickable((By.XPATH, '//*[@id="main-content-wrapper"]/section[1]/div/div[4]/div[3]/button[3]'))
        )
        # print(f'Scraping page \"{page}\"')
        # page += 1
        # Next_button.click()
        # time.sleep(2)
    except:
        print("The 'Next' button is not clickable. we have navigated through all the pages.")
        break
    else:
        page += 1
        print(f'Scraping page \"{page}\"')
        Next_button.click()
        time.sleep(2)
driver.quit()

The page "Yahoo Finance - Stock Market Live, Quotes, Business & Finance News" is ready.
The page "Top Trending Stocks: US stocks with the highest interest today - Yahoo Finance" is ready.
The page "Most Active Stocks: US stocks with the highest trading volume today - Yahoo Finance" is ready.
Scraping page 1
Number of rows Page "1" has is: 25
Scraping page "2"
Number of rows Page "2" has is: 25
Scraping page "3"
Number of rows Page "3" has is: 25
Scraping page "4"
Number of rows Page "4" has is: 25
Scraping page "5"
Number of rows Page "5" has is: 25
Scraping page "6"
Number of rows Page "6" has is: 25
Scraping page "7"
Number of rows Page "7" has is: 25
Scraping page "8"
Number of rows Page "8" has is: 25
Scraping page "9"
Number of rows Page "9" has is: 11
The 'Next' button is not clickable. we have navigated through all the pages.


In [9]:
Data

[{'name': 'ImmunityBio, Inc.',
  'symbol': 'IBRX',
  'price': '5.37',
  'change': '+1.42',
  'volume': '143.449M',
  'market_cap': '5.236B',
  'pr_ratio': '--'},
 {'name': 'Ondas Holdings Inc.',
  'symbol': 'ONDS',
  'price': '13.11',
  'change': '+0.29',
  'volume': '129.999M',
  'market_cap': '5.272B',
  'pr_ratio': '--'},
 {'name': 'NVIDIA Corporation',
  'symbol': 'NVDA',
  'price': '188.16',
  'change': '+1.17',
  'volume': '110.596M',
  'market_cap': '4.581T',
  'pr_ratio': '45.33'},
 {'name': 'Intel Corporation',
  'symbol': 'INTC',
  'price': '47.56',
  'change': '-0.74',
  'volume': '82.901M',
  'market_cap': '227.044B',
  'pr_ratio': '874.66'},
 {'name': 'Plug Power Inc.',
  'symbol': 'PLUG',
  'price': '2.41',
  'change': '+0.15',
  'volume': '70.052M',
  'market_cap': '3.34B',
  'pr_ratio': '--'},
 {'name': 'BigBear.ai Holdings, Inc.',
  'symbol': 'BBAI',
  'price': '6.32',
  'change': '+0.14',
  'volume': '62.026M',
  'market_cap': '2.757B',
  'pr_ratio': '--'},
 {'name': 

In [11]:
len(Data)

211

In [91]:
import pandas as pd
import numpy as np

In [106]:
df = (pd.DataFrame(Data).apply(lambda col:col.str.strip() if col.dtype == "object" else col)
    .assign(price = lambda df_: pd.to_numeric(df_.price),
            change = lambda df_: pd.to_numeric(df_.change.str.replace("+", "")),
            volume = lambda df_: pd.to_numeric(df_.volume.str.replace("M", "")),
            market_cap = lambda df_: pd.to_numeric(df_.market_cap.apply(lambda val: float(val.replace("B", "")) if val[-1] != 'T' else float(val.replace("T", "")) * 1000)),
            pr_ratio = lambda df_ : (
                  df_
                    .pr_ratio
                    .replace('--', np.nan)
                    .str.replace(',', '')
                    .pipe(lambda col: pd.to_numeric(col)))
           )
    
          
    .rename(columns = {
    "price": "price_usd",
    "volume": "volume_M",
    "market_cap": "market_cap_B"
}))

In [112]:
df

,name,symbol,price_usd,change,volume_M,market_cap_B,pr_ratio
0,"ImmunityBio, Inc.",IBRX,5.37,1.42,143.449,5.236,NaN
1,Ondas Holdings Inc.,ONDS,13.11,0.29,129.999,5.272,NaN
2,NVIDIA Corporation,NVDA,188.16,1.17,110.596,4581.000,45.33
3,Intel Corporation,INTC,47.56,-0.74,82.901,227.044,874.66
4,Plug Power Inc.,PLUG,2.41,0.15,70.052,3.340,NaN
...,...,...,...,...,...,...,...
206,XPeng Inc.,XPEV,20.63,-0.25,5.072,19.680,NaN
207,Dell Technologies Inc.,DELL,120.33,0.67,5.066,80.645,15.83
208,"CommScope Holding Company, Inc.",COMM,19.58,0.54,5.054,4.338,14.38
209,"Lucid Group, Inc.",LCID,10.08,0.02,5.040,3.264,NaN


In [115]:
df.to_excel("yahoo-stocks-data.xlsx", index = False)